# R3: Evaluate a Trained Model on the Held-Out Test Set

Rebuilds the deterministic event split from R1/R2, runs a trained model over
the test (held-out) images, aggregates image probabilities to *event* probabilities
(mean softmax across each event's frames), and reports the paper's full
metric suite: argmax confusion matrix, macro F1, MCC, macro AUC, a dummy
baseline, per-class threshold optimization, and calibration (Brier scores).

By default this evaluates the model you trained in R2
(`outputs/lava_lake_classifier_retrained.keras`) if present, else the
published weights shipped in `models/`.


In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os, glob, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve, auc,
                             matthews_corrcoef, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import label_binarize

SEED = 42
retrained = config.OUTPUT_DIR / "lava_lake_classifier_retrained.keras"
MODEL_TO_EVALUATE = retrained if retrained.exists() else config.MODEL_PATH
print("Evaluating:", MODEL_TO_EVALUATE)
if MODEL_TO_EVALUATE == config.MODEL_PATH:
    print("NOTE: these are the published weights -> metrics are an upper bound (see header).")
model = tf.keras.models.load_model(MODEL_TO_EVALUATE, compile=False)


## 1. Rebuild the deterministic split

Identical logic to R2 (same seed, same `sorted()` fix); reuses
`outputs/split_manifest.csv` when R2 already wrote one so the two notebooks
can never drift apart.

In [ ]:
def get_timestamp(path):
    parts = os.path.splitext(os.path.basename(path))[0].split("_")
    return pd.to_datetime(parts[0] + " " + parts[1], format="%Y-%m-%d %H%M%S", errors="coerce")

manifest_path = config.OUTPUT_DIR / "split_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    manifest["path"] = manifest.apply(
        lambda r: str(config.DATASET_DIR / config.CLASS_FOLDERS[r["label"]] / r["path"]), axis=1)
    df_test = manifest[manifest["split"] == "test"].reset_index(drop=True)
    print("Loaded split from R2 manifest.")
else:
    parts = []
    for label_idx, gap, prefix in [(0, config.NO_ERUPTION_GROUPING_SEC, "NO_"),
                                   (1, config.EVENT_GAP_SEC, "SMALL_"),
                                   (2, config.EVENT_GAP_SEC, "LARGE_")]:
        folder = config.DATASET_DIR / config.CLASS_FOLDERS[label_idx]
        d = pd.DataFrame({"path": sorted(glob.glob(str(folder / "*.jpg"))), "label": label_idx})
        d["timestamp"] = d["path"].map(get_timestamp)
        d = d.sort_values("timestamp").reset_index(drop=True)
        new_event = (d["timestamp"].diff() > pd.Timedelta(seconds=gap)) | d["timestamp"].diff().isna()
        d["event_id"] = prefix + new_event.cumsum().astype(str)
        parts.append(d)
    df_all = pd.concat(parts, ignore_index=True)

    def split_ids(ids):
        unique_ids = sorted(set(ids))
        random.seed(SEED)
        random.shuffle(unique_ids)
        n = len(unique_ids)
        n_train, n_val = int(n * 9 / 11), int(n * 1 / 11)
        return unique_ids[n_train + n_val:]

    test_ids = []
    for label in (0, 1, 2):
        test_ids += split_ids(df_all[df_all["label"] == label]["event_id"].unique())
    df_test = df_all[df_all["event_id"].isin(test_ids)].reset_index(drop=True)
    print("Built split directly (no R2 manifest found).")

print(f"Test set: {len(df_test):,} images in {df_test['event_id'].nunique()} events")
print({c: df_test[df_test.label == l]["event_id"].nunique() for c, l in config.CLASS_MAP.items()},
      " <- expect 13/210/349")


## 2. Predict and aggregate to events

An *event's* probability vector is the mean of its frames' softmax outputs.
Aggregating first makes the metrics count each physical eruption once instead
of once per frame.

In [ ]:
def load_img(path):
    img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, [config.IMG_HEIGHT, config.IMG_WIDTH])

ds = (tf.data.Dataset.from_tensor_slices(df_test["path"].values)
      .map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(128).prefetch(tf.data.AUTOTUNE))
probs = model.predict(ds, verbose=1)
for i in range(3):
    df_test[f"prob_{i}"] = probs[:, i]

event_agg = (df_test.groupby("event_id")
             .agg(label=("label", "first"), prob_0=("prob_0", "mean"),
                  prob_1=("prob_1", "mean"), prob_2=("prob_2", "mean"))
             .reset_index())
y_true = event_agg["label"].values
y_prob = event_agg[["prob_0", "prob_1", "prob_2"]].values
y_pred = np.argmax(y_prob, axis=1)
print(f"{len(event_agg)} events aggregated.")


## 3. Global metrics, dummy baseline, and confusion matrix

The *stratified dummy classifier* guesses labels at random in proportion to
the training distribution: the floor any real model must clear. Matthews
Correlation Coefficient (MCC) summarizes the full confusion matrix in one
number robust to class imbalance (+1 perfect, 0 chance).

In [ ]:
y_bin = label_binarize(y_true, classes=[0, 1, 2])
print(f"Macro F1 : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"MCC      : {matthews_corrcoef(y_true, y_pred):.4f}")
print(f"Macro AUC: {roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr'):.4f}")

dummy = DummyClassifier(strategy="stratified", random_state=SEED)
dummy.fit(np.zeros((len(y_true), 1)), y_true)
d_pred = dummy.predict(np.zeros((len(y_true), 1)))
print(f"\nDummy baseline -> macro F1 {f1_score(y_true, d_pred, average='macro'):.4f}, "
      f"MCC {matthews_corrcoef(y_true, d_pred):.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
                       display_labels=config.CLASS_NAMES).plot(cmap="Blues", ax=ax, colorbar=True)
ax.set_title("Event-level confusion matrix (argmax)")
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / "cm_test_events.png", dpi=300)
plt.show()


## 4. Per-class threshold optimization


In [ ]:
optimal = {}
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for class_idx, (name, ax) in enumerate(zip(config.CLASS_NAMES, axes)):
    yb = (y_true == class_idx).astype(int)
    cp = y_prob[:, class_idx]
    grid = np.concatenate([np.linspace(0.01, 0.90, 40), np.linspace(0.91, 0.999, 40)])
    prec, rec, f1v = [], [], []
    for t in grid:
        pb = (cp > t).astype(int)
        prec.append(precision_score(yb, pb, zero_division=0))
        rec.append(recall_score(yb, pb, zero_division=0))
        f1v.append(f1_score(yb, pb, zero_division=0))
    ok = [i for i, p in enumerate(prec) if p >= 0.98]
    k = ok[0] if ok else int(np.argmax(f1v))
    optimal[name] = grid[k]
    print(f"{name:16s}: T = {grid[k]:.4f}  precision {prec[k]:.3f}  recall {rec[k]:.3f}  F1 {f1v[k]:.3f}"
          + ("" if ok else "   (precision floor missed; max-F1 fallback)"))
    ax.plot(grid, f1v, "-", label="F1"); ax.plot(grid, rec, ":", label="Recall")
    ax.plot(grid, prec, "--", color="firebrick", label="Precision")
    ax.axvline(grid[k], color="black", linestyle="-.")
    ax.set_ylim(0.7, 1.0); ax.set_title(name); ax.set_xlabel("Probability threshold"); ax.legend()
plt.tight_layout(); plt.savefig(config.OUTPUT_DIR / "threshold_scan.png", dpi=300); plt.show()

with open(config.OUTPUT_DIR / "threshold_config.txt", "w") as f:
    for name, t in optimal.items():
        f.write(f"{name} Optimal Threshold: {t:.6f}\n")
print("Thresholds ->", config.OUTPUT_DIR / "threshold_config.txt")


## 5. ROC curves and probability calibration

A Brier score is the mean squared error between predicted probability and
the 0/1 outcome: 0 is perfect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ["blue", "orange", "red"]
for i, color in zip(range(3), colors):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{config.CLASS_NAMES[i]} (AUC {auc(fpr, tpr):.3f})")
    brier = brier_score_loss(y_bin[:, i], y_prob[:, i])
    print(f"Brier score ({config.CLASS_NAMES[i]}): {brier:.4f}")
    pt, pp = calibration_curve(y_bin[:, i], y_prob[:, i], n_bins=10)
    axes[1].plot(pp, pt, "s-", color=color, label=f"{config.CLASS_NAMES[i]} (BS {brier:.3f})")
axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("ROC (one-vs-rest)")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate"); axes[0].legend()
axes[1].plot([0, 1], [0, 1], "k:"); axes[1].set_title("Calibration")
axes[1].set_xlabel("Mean predicted probability"); axes[1].set_ylabel("Fraction positive"); axes[1].legend()
plt.tight_layout(); plt.savefig(config.OUTPUT_DIR / "roc_calibration.png", dpi=300); plt.show()
print("\nDone. Next: R4 to classify a new month of imagery, or R5 for the December 2015 analysis.")


## 6. Grad-CAM: where the network is looking

Grad-CAM weights the last convolutional feature maps by the gradient of a class
score with respect to those maps, giving a coarse map of which regions drove the
prediction.

- The heatmap covers the cropped field of view, not the full input. The
  `Cropping2D` layer removes 30 px top and bottom and 40 px each side, so the
  network never sees the outer margin, and the overlay below is drawn on the
  cropped frame accordingly.
- The underlying frame is a three-frame temporal composite, so color is
  motion: gray means the pixel barely changed across the ~4 s window, color
  means it did. Attention concentrated on colored regions is the network
  responding to change rather than to static thermal structure.

Set `SHOW = "errors"` to inspect misclassified frames instead of correct ones.

In [ ]:
CAM_LAYER   = None
N_PER_CLASS = 3          # samples shown per class
SHOW        = "correct"  # "correct" -> confident hits;  "errors" -> misclassified

rng = np.random.default_rng(SEED)

crop_layer = next(l for l in model.layers
                  if l.__class__.__name__ == "Cropping2D")
(top, bottom), (left, right) = crop_layer.get_config()["cropping"]

if CAM_LAYER:
    target = model.get_layer(CAM_LAYER)
else:
    target = [l for l in model.layers
              if l.__class__.__name__ == "Conv2D"][-1]
print(f"Grad-CAM layer: {target.name}   crop: top/bottom {top}/{bottom}, "
      f"left/right {left}/{right}")

try:
    grad_model = tf.keras.Model(model.inputs, [target.output, model.output])
except Exception:
    inp = tf.keras.Input(shape=model.input_shape[1:])
    x, conv_out = inp, None
    for layer in model.layers:
        x = layer(x, training=False)      # training=False keeps the augmentation
        if layer is target:               # layers inert; Keras 3 defaults them True
            conv_out = x
    grad_model = tf.keras.Model(inp, [conv_out, x])

def gradcam(img_batch, class_index):
    """Grad-CAM heatmap over the *cropped* field of view, normalized to [0, 1]."""
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch, training=False)
        tape.watch(conv_out)
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.squeeze(conv_out[0] @ weights[..., tf.newaxis])
    cam = tf.maximum(cam, 0) / (tf.reduce_max(cam) + 1e-10)
    ch, cw = config.IMG_HEIGHT - top - bottom, config.IMG_WIDTH - left - right
    cam = tf.image.resize(cam[..., tf.newaxis], (ch, cw), method="bicubic")
    return tf.squeeze(cam).numpy()

def load_one(path):
    img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, [config.IMG_HEIGHT, config.IMG_WIDTH])

pred_img = df_test[["prob_0", "prob_1", "prob_2"]].to_numpy().argmax(axis=1)
df_test = df_test.assign(_pred=pred_img)

picks = {}
for label in (2, 1, 0):
    pool = df_test[df_test["label"] == label]
    pool = pool[pool["_pred"] != label] if SHOW == "errors" else \
           pool[pool["_pred"] == label]
    if pool.empty:
        print(f"  no '{SHOW}' samples for {config.CLASS_NAMES[label]}")
        continue
    take = min(N_PER_CLASS, len(pool))
    picks[label] = pool.iloc[rng.choice(len(pool), take, replace=False)]

rows, cols = len(picks), 2 * N_PER_CLASS
fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.4 * rows),
                         squeeze=False)
for r, (label, sub) in enumerate(picks.items()):
    for c in range(cols):
        axes[r][c].axis("off")
    for k, row in enumerate(sub.itertuples()):
        img = load_one(row.path)
        heat = gradcam(tf.expand_dims(img, 0), label)
        frame = (img.numpy() / 255.0)[top:config.IMG_HEIGHT - bottom,
                                      left:config.IMG_WIDTH - right]
        a, b = axes[r][2 * k], axes[r][2 * k + 1]
        a.imshow(frame)
        a.set_title(f"{config.CLASS_NAMES[label]}\ncomposite", fontsize=8)
        b.imshow(frame)
        b.imshow(heat, cmap="jet", alpha=0.45)
        p = getattr(row, f"prob_{label}")
        b.set_title(f"P({config.CLASS_NAMES[label].split()[0]}) = {p:.2f}",
                    fontsize=8)

sm = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1))
fig.colorbar(sm, ax=axes, shrink=0.6, label="Relative attention")
fig.suptitle(f"Grad-CAM on the cropped field of view ({SHOW} predictions)",
             fontsize=13)
plt.savefig(config.OUTPUT_DIR / f"gradcam_{SHOW}.png", dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved ->", config.OUTPUT_DIR / f"gradcam_{SHOW}.png")